# Clase 10. Aprendizaje no supervisado y representaciones

**Curso: Advanced Deep Learning**

Este cuaderno es el que ejecutas en directo durante la clase. Alterna explicación breve con código funcionando: cada bloque teórico se apoya en la celda de código que tiene justo debajo.

## Objetivos de aprendizaje

Al terminar esta clase, sabes:

- Explicar por qué el aprendizaje no supervisado importa cuando etiquetar datos tiene un coste alto
- Describir la arquitectura de un autocodificador (codificador más decodificador) y su función de pérdida
- Implementar y entrenar un autocodificador simple en PyTorch sobre el conjunto de datos MNIST
- Comparar visualmente imágenes originales con sus reconstrucciones
- Explorar el espacio latente de dos dimensiones de un autocodificador y relacionarlo con las etiquetas reales de los dígitos

## El coste de etiquetar datos

Casi todo lo que entrenas en aprendizaje profundo necesita etiquetas: una imagen y su clase, un texto y su traducción. Etiquetar cuesta caro. Clasificar el millón de imágenes de ImageNet costó miles de horas de trabajo humano, y en ámbitos como la medicina el etiquetado exige además personal experto.

Mientras tanto, el mundo produce una cantidad casi infinita de datos sin etiquetar: vídeo, audio, texto, registros de sensores. El aprendizaje no supervisado busca aprovechar esos datos sin que nadie los etiquete antes.

## Comprimir para entender

Imagina que describes una fotografía en un mensaje muy corto: "una persona con chaqueta roja, de pie en un paisaje nevado". No envías los píxeles, resumes. Con ese resumen, quien lo lee se hace una idea fiel de la foto, aunque no reconstruya los mismos píxeles.

Un autocodificador (*autoencoder*) hace algo parecido, de forma automática y numérica. Consta de dos piezas:

- Un **codificador**, que comprime la entrada en un vector mucho más pequeño: la representación latente o código
- Un **decodificador**, que toma ese vector y reconstruye la entrada original a partir de él

El cuello de botella, el tamaño reducido del código, impide que la red copie la entrada de forma trivial: la obliga a descubrir qué información merece la pena conservar.

## Construye un autoencoder

Defines el codificador y el decodificador con capas lineales conectadas mediante activaciones ReLU (unidad lineal rectificada). La última capa del decodificador usa una sigmoide, porque los píxeles de MNIST están normalizados entre cero y uno.

Entrenas minimizando el error de reconstrucción entre la imagen original y la reconstruida. Fíjate en que el bucle de entrenamiento nunca usa la etiqueta del dígito.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Usa GPU si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# --- Datos ---
# MNIST: dígitos de 28x28 en escala de grises, valores normalizados entre 0 y 1
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

DIM_ENTRADA = 28 * 28   # cada imagen se aplana a un vector de 784 valores
DIM_LATENTE = 32        # tamaño del cuello de botella: el "resumen" de cada imagen


# --- Definición del autocodificador ---
class Autocodificador(nn.Module):
    def __init__(self, dim_entrada=DIM_ENTRADA, dim_latente=DIM_LATENTE):
        super().__init__()
        # Codificador: comprime progresivamente hasta el cuello de botella
        self.codificador = nn.Sequential(
            nn.Linear(dim_entrada, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, dim_latente),
        )
        # Decodificador: expande de vuelta hasta el tamaño original
        # Sigmoid final porque los píxeles de MNIST están normalizados en [0, 1]
        self.decodificador = nn.Sequential(
            nn.Linear(dim_latente, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, dim_entrada),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.codificador(x)
        x_reconstruida = self.decodificador(z)
        return x_reconstruida, z


modelo = Autocodificador().to(device)
print(modelo)

# --- Función de pérdida y optimizador ---
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


# --- Bucle de entrenamiento ---
def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, _ in loader:  # las etiquetas no se usan: no hace falta supervisión
        imagenes = imagenes.view(imagenes.size(0), -1).to(device)  # aplana a vector

        optimizador.zero_grad()
        reconstruidas, _ = modelo(imagenes)
        perdida = criterio(reconstruidas, imagenes)
        perdida.backward()
        optimizador.step()

        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)


num_epocas = 10
for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    print(f"Época {epoca}/{num_epocas} | pérdida de reconstrucción: {perdida_train:.4f}")

## Original vs. reconstrucción

Comparas diez dígitos de test con su reconstrucción. Espera reconstrucciones algo borrosas, con bordes menos definidos: es la firma típica de una pérdida basada en la distancia entre píxeles.

In [ ]:
modelo.eval()
imagenes_prueba, etiquetas_prueba = next(iter(test_loader))
imagenes_prueba = imagenes_prueba[:10]

with torch.no_grad():
    entrada_plana = imagenes_prueba.view(imagenes_prueba.size(0), -1).to(device)
    reconstruidas, _ = modelo(entrada_plana)
    reconstruidas = reconstruidas.view(-1, 28, 28).cpu()

fig, ejes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    ejes[0, i].imshow(imagenes_prueba[i].squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(reconstruidas[i], cmap="gray")
    ejes[1, i].axis("off")

ejes[0, 0].set_title("Original", loc="left", fontsize=10)
ejes[1, 0].set_title("Reconstruida", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Explora el espacio latente

El código de treinta y dos dimensiones que usaste antes reconstruye bien, pero no lo puedes dibujar directamente. Para ver el espacio latente con tus propios ojos, entrenas una segunda versión del mismo autocodificador con un cuello de botella de solo dos dimensiones.

Después, proyectas imágenes de test en ese espacio de dos dimensiones y coloreas cada punto según el dígito real, aunque el modelo nunca haya visto esa etiqueta durante el entrenamiento.

In [ ]:
# Entrenas una segunda versión del autocodificador con cuello de botella de dos dimensiones
modelo_2d = Autocodificador(dim_latente=2).to(device)
optimizador_2d = torch.optim.Adam(modelo_2d.parameters(), lr=1e-3)

num_epocas_2d = 10
for epoca in range(1, num_epocas_2d + 1):
    perdida_train = entrenar_una_epoca(modelo_2d, train_loader, criterio, optimizador_2d)
    print(f"Época {epoca}/{num_epocas_2d} | pérdida de reconstrucción (2D): {perdida_train:.4f}")

# --- Proyección en el espacio latente ---
modelo_2d.eval()
imagenes_test, etiquetas_test = next(iter(test_loader))
imagenes_test = imagenes_test[:500]
etiquetas_test = etiquetas_test[:500]

with torch.no_grad():
    entrada_plana = imagenes_test.view(imagenes_test.size(0), -1).to(device)
    _, z = modelo_2d(entrada_plana)
    z = z.cpu().numpy()

plt.figure(figsize=(8, 6))
dispersion = plt.scatter(z[:, 0], z[:, 1], c=etiquetas_test.numpy(), cmap="tab10", s=10)
plt.colorbar(dispersion, label="Dígito real")
plt.xlabel("Dimensión latente 1")
plt.ylabel("Dimensión latente 2")
plt.title("Espacio latente de dos dimensiones")
plt.show()

## Resumen de la clase

- El aprendizaje no supervisado aprovecha datos sin etiquetar, evitando el coste del etiquetado manual
- Un autocodificador aprende comprimiendo la entrada en un código latente (codificador) y reconstruyéndola a partir de ese código (decodificador)
- El cuello de botella obliga a la red a priorizar la información más relevante y descartar redundancia
- Entrenas minimizando el error de reconstrucción entre la entrada y su reconstrucción, sin usar ninguna etiqueta
- Un cuello de botella de dos dimensiones permite visualizar el espacio latente directamente, y los dígitos iguales tienden a agruparse aunque el modelo nunca haya visto sus etiquetas

**Abre ahora el cuaderno de práctica `../notebooks_practica/clase10_practica_aprendizaje_no_supervisado.ipynb` y sigue los ejercicios propuestos.**